# Contrastive Explanation Results

Visualizes the Instantiation I ablation results from `scripts/out/journal/summary_contrastive.csv`.

Run `scripts/run_experiments.py` first to generate the data.

In [8]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO = Path('/Users/ssuresh/gambit')
CSV  = REPO / 'scripts' / 'out' / 'journal' / 'summary_contrastive.csv'

df = pd.read_csv(CSV)

METHODS = ['base_evidence', 'naive_contrastive', 'optimized']
METHOD_LABELS = {'base_evidence': 'Base', 'naive_contrastive': 'Naive', 'optimized': 'CDEA'}
EVIDENCE_LABELS = {'gradcam': 'Grad-CAM', 'ig': 'IG'}
datasets = df['dataset'].unique().tolist()
evidence_types = df['evidence'].unique().tolist()

print(f'Datasets: {datasets}')
print(f'Evidence: {evidence_types}')
print(f'Seeds: n={df["n_seeds"].iloc[0]}')
df.head()


Datasets: ['cifar10', 'mnist', 'pets', 'stanford_dogs']
Evidence: ['gradcam', 'ig']
Seeds: n=3


,dataset,evidence,margin_fmt,margin_mean,margin_std,method,n_seeds,overlap_fmt,overlap_mean,overlap_std,sparse_fmt,sparse_mean,sparse_std,suff_fmt,suff_mean,suff_std
0,cifar10,gradcam,-1.4879 ± 0.9384,-1.487932,0.938396,base_evidence,3,0.3087 ± 0.1710,0.308700,0.171038,0.8729 ± 0.2201,0.872917,0.220115,-0.8676 ± 0.8910,-0.867587,0.890966
1,cifar10,gradcam,-1.4880 ± 0.9384,-1.487987,0.938442,naive_contrastive,3,0.1689 ± 0.1237,0.168940,0.123676,0.8729 ± 0.2201,0.872917,0.220115,-0.8678 ± 0.8911,-0.867792,0.891140
2,cifar10,gradcam,-1.2053 ± 0.6936,-1.205300,0.693632,optimized,3,0.0075 ± 0.0049,0.007468,0.004851,0.8832 ± 0.2289,0.883162,0.228883,-0.6850 ± 0.7328,-0.684972,0.732807
3,cifar10,ig,-1.3987 ± 1.1125,-1.398737,1.112550,base_evidence,3,0.4994 ± 0.0033,0.499427,0.003285,1.0000,1.000000,0.000000,-0.8611 ± 0.9149,-0.861119,0.914879
4,cifar10,ig,-1.3983 ± 1.1121,-1.398298,1.112141,naive_contrastive,3,0.2961 ± 0.0187,0.296131,0.018701,1.0000,1.000000,0.000000,-0.8638 ± 0.9172,-0.863777,0.917164


## 1. Main Results Table

Optimized (CDEA) metrics vs base evidence, one row per (dataset, evidence).

In [9]:
base = df[df['method'] == 'base_evidence'].set_index(['dataset', 'evidence'])
opt  = df[df['method'] == 'optimized'].set_index(['dataset', 'evidence'])

rows = []
for ds in datasets:
    for ev in evidence_types:
        if (ds, ev) not in base.index or (ds, ev) not in opt.index:
            continue
        b = base.loc[(ds, ev)]
        o = opt.loc[(ds, ev)]
        ov_red = (b['overlap_mean'] - o['overlap_mean']) / max(abs(b['overlap_mean']), 1e-8) * 100
        sp_red = (b['sparse_mean'] - o['sparse_mean']) / max(abs(b['sparse_mean']), 1e-8) * 100
        rows.append({
            'Dataset': ds, 'Evidence': EVIDENCE_LABELS.get(ev, ev),
            'Suff ↑': f"{o['suff_mean']:.4f}",
            'Margin ↑': f"{o['margin_mean']:.4f}",
            'Overlap ↓': f"{o['overlap_mean']:.4f}",
            'Overlap ↓% vs Base': f"{ov_red:+.1f}%",
            'Sparse ↓': f"{o['sparse_mean']:.4f}",
            'Sparse ↓% vs Base': f"{sp_red:+.1f}%",
        })

main_table = pd.DataFrame(rows)
main_table.style.set_caption('CDEA (optimized) — main results').hide(axis='index')

Dataset,Evidence,Suff ↑,Margin ↑,Overlap ↓,Overlap ↓% vs Base,Sparse ↓,Sparse ↓% vs Base
cifar10,Grad-CAM,-0.6850,-1.2053,0.0075,+97.6%,0.8832,-1.2%
cifar10,IG,-0.6551,-1.0987,0.0486,+90.3%,0.9121,+8.8%
mnist,Grad-CAM,-1.0995,-2.0629,0.0090,+97.8%,0.9968,+0.3%
mnist,IG,-1.0818,-2.0766,0.0468,+97.3%,0.8969,+10.3%
pets,Grad-CAM,0.6829,0.4443,0.0001,+93.5%,1.0187,-2.2%
pets,IG,0.7266,0.5373,0.0050,+91.5%,1.0198,-2.0%
stanford_dogs,Grad-CAM,-4.3445,-1.3176,0.0164,+96.7%,1.0182,-1.8%
stanford_dogs,IG,-4.3489,-1.3482,0.0215,+96.2%,1.0128,-1.3%


## 2. Overlap by Method — Grouped Bar Chart

Shows how CDEA drives overlap toward zero vs base evidence and naive contrastive.  
Overlap is an unnormalized pairwise dot-product sum — lower is better.

In [10]:
METHOD_COLORS = {
    'base_evidence':    '#6baed6',
    'naive_contrastive':'#fd8d3c',
    'optimized':        '#31a354',
}

fig = make_subplots(
    rows=1, cols=len(evidence_types),
    subplot_titles=[EVIDENCE_LABELS.get(e, e) for e in evidence_types],
    shared_yaxes=False,
)

for col_idx, ev in enumerate(evidence_types, start=1):
    for method in METHODS:
        sub = df[(df['evidence'] == ev) & (df['method'] == method)]
        means = [sub[sub['dataset'] == ds]['overlap_mean'].values[0]
                 if not sub[sub['dataset'] == ds].empty else 0
                 for ds in datasets]
        stds  = [sub[sub['dataset'] == ds]['overlap_std'].values[0]
                 if not sub[sub['dataset'] == ds].empty else 0
                 for ds in datasets]
        fig.add_trace(go.Bar(
            name=METHOD_LABELS[method],
            x=datasets, y=means,
            error_y=dict(type='data', array=stds, visible=True),
            marker_color=METHOD_COLORS[method],
            opacity=0.85,
            showlegend=(col_idx == 1),
        ), row=1, col=col_idx)
        fig.update_yaxes(title_text='Overlap (↓ better)', row=1, col=col_idx)

fig.update_layout(
    title_text='Pairwise Mask Overlap by Method', title_x=0.5,
    barmode='group', height=420, width=500 * len(evidence_types),
    legend=dict(orientation='h', yanchor='bottom', y=1.05, x=0.3),
    plot_bgcolor='white',
)
out = REPO / 'examples' / 'out' / 'plot_overlap_by_method.html'
fig.write_html(str(out))
fig.show()
print('Saved to', out)


Saved to /Users/ssuresh/gambit/examples/out/plot_overlap_by_method.html


## 3. Overlap Reduction % — Heatmap

Percent overlap reduction of CDEA over base evidence.  
Higher is better — the headline result.

In [11]:
heat = np.full((len(datasets), len(evidence_types)), float('nan'))
for i, ds in enumerate(datasets):
    for j, ev in enumerate(evidence_types):
        b = df[(df['dataset']==ds)&(df['evidence']==ev)&(df['method']=='base_evidence')]
        o = df[(df['dataset']==ds)&(df['evidence']==ev)&(df['method']=='optimized')]
        if not b.empty and not o.empty:
            b_ov = b['overlap_mean'].values[0]
            o_ov = o['overlap_mean'].values[0]
            heat[i, j] = (b_ov - o_ov) / max(abs(b_ov), 1e-8) * 100

text_vals = [[f'{heat[i,j]:.1f}%' if not np.isnan(heat[i,j]) else 'n/a'
              for j in range(len(evidence_types))] for i in range(len(datasets))]

fig = go.Figure(go.Heatmap(
    z=heat,
    x=[EVIDENCE_LABELS.get(e, e) for e in evidence_types],
    y=datasets,
    colorscale='YlGn', zmin=0, zmax=100,
    text=text_vals, texttemplate='%{text}', textfont={'size': 13},
    colorbar=dict(title='Reduction %', thickness=15),
    hovertemplate='%{y} / %{x}<br>Overlap reduction: %{z:.1f}%<extra></extra>',
))
fig.update_layout(
    title_text='Overlap Reduction: CDEA vs Base Evidence', title_x=0.5,
    xaxis_title='Evidence', yaxis_title='Dataset',
    height=max(300, len(datasets)*80 + 150), width=max(500, len(evidence_types)*200 + 200),
    plot_bgcolor='white',
)
out = REPO / 'examples' / 'out' / 'plot_overlap_reduction_heatmap.html'
fig.write_html(str(out))
fig.show()
print('Saved to', out)


Saved to /Users/ssuresh/gambit/examples/out/plot_overlap_reduction_heatmap.html


## 4. Sufficiency–Overlap Tradeoff

Each point is a (dataset, evidence, method) combination.  
Ideal: high sufficiency (right) + low overlap (bottom) — bottom-right corner.  
CDEA should cluster bottom-right vs base/naive clustering top-right.

In [12]:
EV_COLORS = {'gradcam': '#2166ac', 'ig': '#d6604d'}
MARKERS   = {'base_evidence': 'circle', 'naive_contrastive': 'square', 'optimized': 'star'}
SIZES     = {'base_evidence': 8, 'naive_contrastive': 10, 'optimized': 16}
OPACITIES = {'base_evidence': 0.4, 'naive_contrastive': 0.65, 'optimized': 1.0}

fig = go.Figure()
for method in METHODS:
    for ev in evidence_types:
        sub = df[(df['method']==method) & (df['evidence']==ev)]
        if sub.empty:
            continue
        fig.add_trace(go.Scatter(
            x=sub['suff_mean'], y=sub['overlap_mean'],
            mode='markers+text',
            text=sub['dataset'],
            textposition='top center',
            textfont=dict(size=9),
            marker=dict(
                symbol=MARKERS[method],
                color=EV_COLORS.get(ev, 'gray'),
                size=SIZES[method],
                opacity=OPACITIES[method],
                line=dict(width=1, color='white') if method=='optimized' else dict(width=0),
            ),
            name=f'{METHOD_LABELS[method]} / {EVIDENCE_LABELS.get(ev, ev)}',
            hovertemplate='%{text}<br>Suff: %{x:.4f}<br>Overlap: %{y:.4f}<extra></extra>',
        ))

fig.update_layout(
    title_text='Sufficiency–Overlap Tradeoff (CDEA=star, Base=circle, Naive=square)',
    title_x=0.5,
    xaxis_title='Sufficiency ↑', yaxis_title='Overlap ↓',
    height=500, width=750,
    legend=dict(font=dict(size=9)),
    plot_bgcolor='white',
)
fig.update_xaxes(gridcolor='rgba(0,0,0,0.08)')
fig.update_yaxes(gridcolor='rgba(0,0,0,0.08)')
out = REPO / 'examples' / 'out' / 'plot_suff_overlap_tradeoff.html'
fig.write_html(str(out))
fig.show()
print('Saved to', out)


Saved to /Users/ssuresh/gambit/examples/out/plot_suff_overlap_tradeoff.html


## 5. Margin Improvement — CDEA vs Base

Contrastive margin = `z_k − max(z_foil)` under the keep intervention.  
Bars show the delta: `margin(CDEA) − margin(base)`. Positive = CDEA improves contrastive separation.

In [13]:
EV_PALETTE = ['#2166ac', '#d6604d', '#4daf4a', '#984ea3']

fig = go.Figure()
for j, ev in enumerate(evidence_types):
    deltas = []
    for ds in datasets:
        b = df[(df['dataset']==ds)&(df['evidence']==ev)&(df['method']=='base_evidence')]
        o = df[(df['dataset']==ds)&(df['evidence']==ev)&(df['method']=='optimized')]
        deltas.append(
            o['margin_mean'].values[0] - b['margin_mean'].values[0]
            if not b.empty and not o.empty else 0
        )
    fig.add_trace(go.Bar(
        name=EVIDENCE_LABELS.get(ev, ev),
        x=datasets, y=deltas,
        marker_color=EV_PALETTE[j % len(EV_PALETTE)],
        text=[f'{v:+.4f}' for v in deltas],
        textposition='outside',
    ))

fig.add_hline(y=0, line_color='black', line_width=1)
fig.update_layout(
    title_text='Contrastive Margin Improvement: CDEA vs Base Evidence', title_x=0.5,
    barmode='group',
    yaxis_title='Margin delta (CDEA − Base) ↑',
    height=420, width=max(500, len(datasets)*160),
    legend=dict(orientation='h', yanchor='bottom', y=1.05),
    plot_bgcolor='white',
)
fig.update_yaxes(gridcolor='rgba(0,0,0,0.08)', zeroline=True, zerolinecolor='black')
out = REPO / 'examples' / 'out' / 'plot_margin_improvement.html'
fig.write_html(str(out))
fig.show()
print('Saved to', out)


Saved to /Users/ssuresh/gambit/examples/out/plot_margin_improvement.html


## 6. Per-Batch Metric Distribution

Box plots of per-batch overlap across methods, for each (dataset, evidence) combination.  
Shows variance across batches — useful for understanding optimizer stability.

In [14]:
import glob

per_batch_frames = []
pattern = str(REPO / 'scripts' / 'out' / 'ablation_*_seed*_per_batch.csv')
for fpath in sorted(glob.glob(pattern)):
    fname = Path(fpath).stem
    parts = fname.split('_')
    if len(parts) < 4:
        continue
    dataset_name  = parts[1]
    evidence_name = parts[2]
    seed_tag      = parts[3]
    tmp = pd.read_csv(fpath)
    tmp['dataset']  = dataset_name
    tmp['evidence'] = evidence_name
    tmp['seed']     = seed_tag
    per_batch_frames.append(tmp)

if not per_batch_frames:
    print('No per-batch CSVs found. Run run_experiments.py first.')
else:
    pb = pd.concat(per_batch_frames, ignore_index=True)
    combos = pb[['dataset', 'evidence']].drop_duplicates().values.tolist()
    METHOD_COLORS2 = {'base_evidence':'#6baed6', 'naive_contrastive':'#fd8d3c', 'optimized':'#31a354'}

    fig = make_subplots(
        rows=1, cols=len(combos),
        subplot_titles=[f'{ds}/{EVIDENCE_LABELS.get(ev,ev)}' for ds, ev in combos],
    )
    for col_idx, (ds, ev) in enumerate(combos, start=1):
        sub = pb[(pb['dataset']==ds) & (pb['evidence']==ev)]
        for method in METHODS:
            vals = sub[sub['method']==method]['overlap'].dropna().tolist()
            fig.add_trace(go.Box(
                y=vals, name=METHOD_LABELS[method],
                marker_color=METHOD_COLORS2[method],
                showlegend=(col_idx==1),
                boxmean=True,
            ), row=1, col=col_idx)
        fig.update_yaxes(title_text='Overlap ↓', row=1, col=col_idx)

    fig.update_layout(
        title_text='Per-Batch Overlap Distribution by Method', title_x=0.5,
        height=420, width=max(500, 350 * len(combos)),
        boxmode='group', plot_bgcolor='white',
        legend=dict(orientation='h', yanchor='bottom', y=1.05),
    )
    out = REPO / 'examples' / 'out' / 'plot_overlap_distribution.html'
    fig.write_html(str(out))
    fig.show()
    print('Saved to', out)


Saved to /Users/ssuresh/gambit/examples/out/plot_overlap_distribution.html
